# SFT + QLoRA 微调（4-bit 基座 + LoRA 适配器）

与 `5.1 SFT+Full_Fine-tuning.ipynb` 用**同一基座、同一数据集、同一预处理、同一有效 batch**，唯一变量是训练单元：

| | 5.1 全参 SFT | 5.2 QLoRA |
|---|---|---|
| 基座权重 | 全量参与训练 | **4-bit(nf4) 量化后冻结** |
| 可训练参数 | 91.37M（100%） | LoRA 适配器 2.52M（2.69%） |
| 存档 | 365MB | adapter 权重本体 10MB（merge 后才是完整模型） |

同一份代码在两种设备上通用，自动按设备切换配置：

| | 服务器（RTX 4090 24GB / CUDA） | Mac（M5 / MPS） |
|---|---|---|
| 计算精度 | **bf16**（Ampere 原生支持，无需 loss scaling） | fp32（实测 fp16 无收益） |
| 优化器 | **paged_adamw_8bit**（QLoRA 标准配方） | AdamW |
| batch | 8×2 = 有效 16，eval 8 | 4×4 = 有效 16，eval 2 |
| dataloader | 4 worker + pin_memory | 0 worker |
| 缓存补丁 | **不用**（CUDA 分配器自己复用，强行 `empty_cache` 更慢） | 每 10 步 `torch.mps.empty_cache()` |




### 1.依赖导入与路径配置

In [1]:
import json
import os
import statistics
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
import pandas as pd
from tqdm.auto import tqdm

# 路径
BASE_MODEL_PATH = "/root/wjx/动手学习大模型/LLM/动手学习大模型/models/bookcorpus-pretrain-91m-1513steps"  # 基座模型
ALPACA_PATH = "/root/wjx/动手学习大模型/LLM/动手学习大模型/datasets/alpaca-chinese-52k-v3.json"           # 微调数据集
ADAPTER_DIR = "/root/wjx/动手学习大模型/LLM/动手学习大模型/models/bookcorpus-qlora-alpaca-zh-91m"         # LoRA 适配器（配基座使用）
MERGED_DIR = "/root/wjx/动手学习大模型/LLM/动手学习大模型/models/bookcorpus-qlora-alpaca-zh-91m-merged"   # 合并后的完整模型
CEVAL_DIR = "/root/wjx/动手学习大模型/LLM/动手学习大模型/datasets/Evaluate/ceval"                         # C-Eval 评测集

# 规模
MAX_LEN = 512           # 每条样本的 token 上限，数据集超过 512 的只有 63/51952 条（0.12%）
MAX_ITERATIONS = None        # 50 = 快速验证，None = 跑满 1 epoch
# 训练参数
BATCH_SIZE = 16
GRAD_ACCUM = 1          # 梯度累积迭代数


libgomp: Invalid value for environment variable OMP_NUM_THREADS


### 2.选择运行设备

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备：{DEVICE}")

使用设备：cuda


### 3.加载数据集

In [3]:
# 加载数据集
with open(ALPACA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# 将指令与输出逐条转成 {prompt, response} 格式
records = []
for x in raw:
    instruction = (x["zh_instruction"] or "").strip()
    extra = (x["zh_input"] or "").strip()
    output = (x["zh_output"] or "").strip()
    if not instruction or not output:
        continue
    records.append({"prompt": f"{instruction}\n{extra}" if extra else instruction,
                    "response": output})
print(f"有效中文样本 {len(records)} 条（原始 {len(raw)} 条）")

# 将 records 列表转成 HuggingFace Dataset，并划分训练/验证集
ds = Dataset.from_list(records).train_test_split(test_size=0.02, seed=42)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
# 分词后长度不整齐，需要padding的部分用eos填充
tokenizer.pad_token = tokenizer.eos_token

# 把文本格式的问答数据，转换成模型训练时真正能吃的数字格式（token id），并且让模型只学"答案"部分
def tokenize_function(examples):
    # 模型输入
    input_ids_list = []
    # 标签，即答案
    labels_list = []
    for prompt, response in zip(examples["prompt"], examples["response"]):
        # 拼接输入
        prompt_text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        # 拼接正确输出
        full_ids = tokenizer(prompt_text + response + "<|eot_id|>", add_special_tokens=False)["input_ids"]
        # 构造labels:问题 + 答案
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        input_ids_list.append(full_ids[:MAX_LEN])
        labels_list.append(labels[:MAX_LEN])
    # 填充进样本
    examples["input_ids"] = input_ids_list
    examples["labels"] = labels_list
    return examples


# 分词
tokenized_ds = ds.map(tokenize_function, batched=True, remove_columns=ds["train"].column_names)
# 过滤无效样本
tokenized_ds = tokenized_ds.filter(lambda x: any(l != -100 for l in x["labels"]))

# 统计数据集
lengths = [len(x) for x in tokenized_ds["train"]["input_ids"]]
supervised = [sum(1 for l in x if l != -100) for l in tokenized_ds["train"]["labels"]]
print(f"train {len(tokenized_ds['train'])} 条 / eval {len(tokenized_ds['test'])} 条")
print(f"序列长度 mean {statistics.mean(lengths):.0f} max {max(lengths)}；"
      f"答案 token 占比 {sum(supervised) / sum(lengths):.0%}")

# 整理器:将长度不一的单挑样本，拼成长度对齐的一批样本
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=64)

有效中文样本 51952 条（原始 52002 条）


Map:   0%|          | 0/50912 [00:00<?, ? examples/s]

Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50912 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1040 [00:00<?, ? examples/s]

train 50911 条 / eval 1040 条
序列长度 mean 94 max 512；答案 token 占比 7%


### 4.微调训练

#### 4.1 训练配置

In [4]:
# 第一步：加载模型并量化到 4-bit
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",              # 4-bit
        bnb_4bit_use_double_quant=True,         # 量化常数再量化一次，每参数再省 ~0.4 bit
        bnb_4bit_compute_dtype=torch.bfloat16,  # 反量化后的计算精度
    ),
    device_map={"": DEVICE},
)
model.config.use_cache = False                  # 训练阶段不需要 KV cache

# 第二步：LoRA 配置
peft_config = LoraConfig(
    r=32,                   # 秩
    lora_alpha=16,          # 缩放系数
    lora_dropout=0.05,      # lora 层的 dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],   # 要插入 LoRA 的模型层
)

# 第三步：计算学习率更新的预热次数与迭代数
total_iterations = MAX_ITERATIONS or -(-len(tokenized_ds["train"]) // (BATCH_SIZE * GRAD_ACCUM))
warmup = max(1, int(0.03 * total_iterations))
print(f"训练数据 {len(tokenized_ds['train'])} 条，总迭代数 {total_iterations}，预热 {warmup} 次")


训练数据 50911 条，总迭代数 3182，预热 95 次


#### 4.2 训练

In [5]:
# 训练配置
training_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    max_seq_length=MAX_LEN,
    num_train_epochs=1,
    max_steps=MAX_ITERATIONS if MAX_ITERATIONS else -1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2.0e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup,
    optim="paged_adamw_8bit",                   # QLoRA 标准配方：优化器状态分页到 CPU，省显存
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=200,
    per_device_eval_batch_size=8,               # 128k 词表 logits 的显存随 batch×长度线性涨
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    seed=42,
    bf16=torch.cuda.is_bf16_supported(),
    dataloader_num_workers=4,
    report_to=[],
)
# 构建训练器
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=collator,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# 执行训练
trainer.train()

trainable params: 2,523,136 || all params: 93,889,024 || trainable%: 2.6874
显存已分配 0.14GiB | 预留 0.39GiB


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss,Validation Loss
200,10.926000,10.873804
400,10.805100,10.732963
600,10.687200,10.691711
800,10.613300,10.639752
1000,10.600800,10.587493
1200,10.500700,10.517533
1400,10.461700,10.462201
1600,10.455000,10.398074
1800,10.423800,10.357247
2000,10.355700,10.323524


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss,Validation Loss
200,10.926000,10.873804
400,10.805100,10.732963
600,10.687200,10.691711
800,10.613300,10.639752
1000,10.600800,10.587493
1200,10.500700,10.517533
1400,10.461700,10.462201
1600,10.455000,10.398074
1800,10.423800,10.357247
2000,10.355700,10.323524


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=3182, training_loss=10.549790450119508, metrics={'train_runtime': 542.3519, 'train_samples_per_second': 93.871, 'train_steps_per_second': 5.867, 'total_flos': 2329885147594752.0, 'train_loss': 10.549790450119508, 'epoch': 1.0})

#### 4.4 保存 LoRA

In [8]:
# 打印模型里有多少参数是可训练的、多少是冻结的。
trainer.model.print_trainable_parameters()
print(f"显存已分配 {torch.cuda.memory_allocated() / 2**30:.2f}GiB | 预留 {torch.cuda.memory_reserved() / 2**30:.2f}GiB")

# 保存模型
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("已保存适配器:", ADAPTER_DIR)

trainable params: 0 || all params: 91,365,888 || trainable%: 0.0000
显存已分配 0.18GiB | 预留 0.33GiB
已保存适配器: /root/wjx/动手学习大模型/LLM/动手学习大模型/models/bookcorpus-qlora-alpaca-zh-91m


### 5.合并存档与自检
- 把 LoRA 的"小补丁"焊进基座，得到一个"普通模型"

### 5.1 合并 lora 头到基模
- 合并是为了工程部署，也可以只用推理 lora
- 合并后推理计算的更快,合并前计算两次：输出 = W·x + BA·x 合并后计算一次：输出 = W'·x,但是速度不会差别很多

In [ ]:
# 将 LoRA 头合并到基模中
merged = trainer.model.merge_and_unload()       # 把 LoRA 的 BA 矩阵融进基座权重，卸载 LoRA 结构，返回一个普通的 LlamaForCausalLM
merged.config.use_cache = True                  # 打开 KV cache，用于推理
# 保存模型与分词器
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# 对比模型大小
adapter_mb = os.path.getsize(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")) / 1e6
merged_mb = sum(os.path.getsize(os.path.join(MERGED_DIR, f)) for f in os.listdir(MERGED_DIR)) / 1e6
print(f"LoRA 权重 {adapter_mb:.1f}MB | merged 完整模型 {merged_mb:.1f}MB（对比 5.1 全参存档 365MB）")

### 5.2 测试微调后模型效果

In [9]:
# 测试微调后模型对话
def chat(model, prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],  tokenize=False, add_generation_prompt=True)
    ids = torch.tensor([tokenizer(text, add_special_tokens=False)["input_ids"]], device=DEVICE)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=60, do_sample=True, temperature=0.7,top_k=50, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


# 对比基模与微调后效果
for name, path in [("基座 91M", BASE_MODEL_PATH), ("QLoRA(merged)", MERGED_DIR)]:
    m = AutoModelForCausalLM.from_pretrained(path).to(DEVICE).eval()
    m.config.use_cache = True
    for p in ["给出保持健康的三个建议。", "三原色是什么？"]:
        print(f"[{name}] {p}\n  -> {chat(m, p)}\n")
    del m

LoRA 权重 0.0MB | merged 完整模型 162.0MB（对比 5.1 全参存档 365MB）
[基座 91M] 给出保持健康的三个建议。
  -> , and its the two men who 'd gone with his parents to be here, and they were definitely a good man.you're already getting married, and i know you're okay with me.`` yeah.she's been a man, but i'm sure



Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


[基座 91M] 三原色是什么？
  -> .`` no. ''`` i'm sorry i'm not. ''`` i hate to talk about it, but i'm trying to get out of your life.the way she 'd gotten to him for the last time in her life.but she just

[QLoRA(merged)] 给出保持健康的三个建议。
  -> 1.000 3. 2. 2.1.11.2. 1.13. 8edith. 1.with 11 3. 3. 2. 6. 2. 5. 6. 3. 2

[QLoRA(merged)] 三原色是什么？
  -> _ASYNCith. 4. 10. 2. 7. 5, 3.6. 5 and t.4. 4. 2.1. 5c. 2. ellen. ellen. 5ra12, 2n.



### 6.C-Eval 评估（基座 vs QLoRA）

- 评测口径:10 科 × 60 题 = 600 题、5-shot、比较 A/B/C/D 的 log-prob，

In [10]:
# 10 科 × 60 题 = 600 题（随机猜 25%）
SUBJECTS = {                    # 10 科 × 60 题 = 600 题（随机猜 25%）
    "high_school_chinese": "高中语文", "high_school_history": "高中历史",
    "high_school_geography": "高中地理", "chinese_language_and_literature": "中国语言文学",
    "law": "法学", "college_physics": "大学物理",
    "high_school_mathematics": "高中数学", "computer_network": "计算机网络",
    "art_studies": "艺术学", "logic": "逻辑学",
}
N_SHOT = 5                      # few-shot 示例数
MAX_Q_PER_SUBJECT = 60          # 每科题数（test 每科 166~298 题；全量改 None）
MAX_EVAL_LEN = 1024             # = 基座 max_position_embeddings
OPTIONS = ["A", "B", "C", "D"]
OPTION_IDS = [tokenizer.convert_tokens_to_ids(c) for c in OPTIONS]


def prompt_ids(text):
    """套上训练时的对话模板再编码，保证评测格式与训练一致"""
    rendered = tokenizer.apply_chat_template([{"role": "user", "content": text}],
                                             tokenize=False, add_generation_prompt=True)
    return tokenizer(rendered, add_special_tokens=False)["input_ids"]


def q_block(row):
    """题干 + 四个选项"""
    return str(row["question"]).strip() + "".join(f"\n{c}. {str(row[c]).strip()}" for c in OPTIONS)


def build_prompt(subject_zh, dev, row):
    """题干放最后；few-shot 示例超长就从最早的开始丢，避免题干被截断"""
    head = f"以下是中国关于{subject_zh}考试的单项选择题，请选出其中的正确答案。\n\n"
    shots = [q_block(ex) + f"\n答案：{str(ex['answer']).strip()}" for _, ex in dev.head(N_SHOT).iterrows()]
    while shots and len(prompt_ids(head + "".join(s + "\n\n" for s in shots) + q_block(row))) > MAX_EVAL_LEN:
        shots.pop(0)
    return head + "".join(s + "\n\n" for s in shots) + q_block(row) + "\n答案："


# 1) 一次建好全部 prompt，两个模型共用
items = []                      # [(科目, prompt, 正确答案)]
for subject, zh in SUBJECTS.items():
    dev = pd.read_parquet(f"{CEVAL_DIR}/{subject}/dev-00000-of-00001.parquet")
    test = pd.read_parquet(f"{CEVAL_DIR}/{subject}/test-00000-of-00001.parquet")
    items += [(subject, build_prompt(zh, dev, r), str(r["answer"]).strip())
              for _, r in (test.head(MAX_Q_PER_SUBJECT) if MAX_Q_PER_SUBJECT else test).iterrows()]
print(f"评测集 {len(items)} 题 / {len(SUBJECTS)} 科 | device={DEVICE}")


# 2) 单题打分：取最后一个位置上 A/B/C/D 的 log-prob，最大者即预测
@torch.no_grad()
def predict(model, prompt):
    ids = torch.tensor([prompt_ids(prompt)], device=model.device)
    logp = torch.log_softmax(model(ids).logits[0, -1].float(), dim=-1)
    return OPTIONS[int(logp[OPTION_IDS].argmax())]


# 3) 基座 vs QLoRA，逐科对比
results = {}
for name, path in [("基座 91M", BASE_MODEL_PATH), ("QLoRA", MERGED_DIR)]:
    model = AutoModelForCausalLM.from_pretrained(path).to(DEVICE).eval()
    preds = [predict(model, p) for _, p, _ in tqdm(items, desc=name, leave=False)]
    acc = pd.DataFrame([{"subject": s, "ok": pr == a} for (s, _, a), pr in zip(items, preds)]) \
            .groupby("subject")["ok"].mean().rename(name)
    results[name] = acc
    print(f"[{name}] 平均准确率 {acc.mean():.1%}")

pd.concat(results.values(), axis=1).map(lambda x: f"{x:.1%}")


评测集 600 题 / 10 科 | device=cuda


基座 91M:   0%|          | 0/600 [00:00<?, ?it/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


[基座 91M] 平均准确率 24.7%


QLoRA:   0%|          | 0/600 [00:00<?, ?it/s]

[QLoRA] 平均准确率 28.5%


,基座 91M,QLoRA
subject,,
art_studies,25.0%,20.0%
chinese_language_and_literature,33.3%,23.3%
college_physics,26.7%,36.7%
computer_network,20.0%,35.0%
high_school_chinese,18.3%,25.0%
high_school_geography,25.0%,33.3%
high_school_history,16.7%,30.0%
high_school_mathematics,20.0%,26.7%
law,28.3%,36.7%
